# Environment smoke test
Setup verification only: check the isolated Python kernel, package imports, small numerical operations, and plot rendering. No curriculum or datasets.

In [ ]:
import os
from pathlib import Path
import site
import sys

assert sys.version_info >= (3, 11), sys.version
assert sys.prefix != sys.base_prefix, 'Select the ML Academy .venv kernel.'
assert not site.ENABLE_USER_SITE
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
expected_python = Path(os.environ.get('ML_ACADEMY_EXPECTED_PYTHON', project_root / '.venv' / 'Scripts' / 'python.exe'))
assert Path(sys.executable).resolve() == expected_python.resolve(), sys.executable
print('Python:', sys.version.split()[0])
print('Kernel:', sys.executable)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import sklearn
from importlib.metadata import version

for module in (np, pd, matplotlib, sklearn):
    assert Path(module.__file__).resolve().is_relative_to(Path(sys.prefix).resolve()), module.__file__
for package in ('notebook', 'ipykernel', 'nbclient', 'nbformat', 'numpy', 'pandas', 'matplotlib', 'scikit-learn'):
    print(f'{package}: {version(package)}')
values = np.array([1.0, 2.0, 3.0])
frame = pd.DataFrame({'value': values})
assert np.isclose(values.sum(), 6.0)
assert np.isclose(frame['value'].mean(), 2.0)
print('NumPy and pandas checks passed.')

In [ ]:
from sklearn.preprocessing import StandardScaler

scaled = StandardScaler().fit_transform(values.reshape(-1, 1))
np.testing.assert_allclose(scaled.mean(axis=0), [0.0], atol=1e-12)
np.testing.assert_allclose(scaled.std(axis=0), [1.0], atol=1e-12)
print('scikit-learn numerical check passed.')

In [ ]:
import io
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

figure, axis = plt.subplots(figsize=(4, 2.5))
axis.plot(values, values ** 2, marker='o')
axis.set(title='Environment check', xlabel='x', ylabel='x squared')
figure.tight_layout()
buffer = io.BytesIO()
figure.savefig(buffer, format='png')
plt.close(figure)
png = buffer.getvalue()
assert png.startswith(b'\x89PNG\r\n\x1a\n')
display(Image(data=png))
print('Environment smoke test passed.')